# Dice Success Distribution Calculator

**Rules:**
* Roll `N` d20 dice.
* Target number `T` (between 2 and 20).
* If roll < `T`, count as 1 success.
* If roll == `T`, count as 2 successes.
* If roll > `T`, count as 0 successes.

This notebook calculates the exact probability distribution for the total number of successes across varying numbers of dice.

In [14]:
import numpy as np

def calculate_success_distribution(num_dice_array, target_number):
    if not (2 <= target_number <= 20):
        print("Target number must be between 2 and 20.")
        return
        
    if isinstance(num_dice_array, int):
        num_dice_array = [num_dice_array]
        
    # Sort the array so columns go from lowest to highest dice count
    num_dice_array = sorted(num_dice_array)

    # Probabilities for a single die (from 20 possible outcomes)
    p_0 = (20 - target_number) / 20.0     # Rolls > target
    p_1 = (target_number - 1) / 20.0      # Rolls < target
    p_2 = 1.0 / 20.0                      # Roll == target
    
    single_die_poly = np.array([p_0, p_1, p_2])
    
    # Store results to build the table
    results_exact = {}
    results_cumulative = {}
    max_successes_possible = 0
    
    for num_dice in num_dice_array:
        # Convolve for `num_dice` using np.polynomial.polynomial.polypow
        dist_poly = np.polynomial.polynomial.polypow(single_die_poly, num_dice)
        percentages = dist_poly * 100
        results_exact[num_dice] = percentages
        
        # Compute cumulative probabilities (X successes OR BETTER)
        cumul = np.zeros_like(percentages)
        for s in range(len(percentages)):
            cumul[s] = np.sum(percentages[s:])
        results_cumulative[num_dice] = cumul
        
        max_successes_possible = max(max_successes_possible, len(dist_poly) - 1)
        
    # --- EXACT PROBABILITIES TABLE ---
    print(f"Target Number: {target_number} (EXACT %)")
    print("=" * (11 + 11 * len(num_dice_array)))
    
    # Header
    # The first column is successes, followed by X Dice columns.
    # Every block (including the separator) needs to map down to strictly 8 visual chars + " | "
    header = f"{'Succes':<8} | "
    for num_dice in num_dice_array:
        # E.g. "5 Dice" padded to 8 chars
        h_col = f"{num_dice} Dice"
        header += f"{h_col:>8} | "
        
    print(header)
    print("-" * len(header))
    
    # Print the exact rows
    for s in range(max_successes_possible + 1):
        row_str = f"{s:<8} | "
        for num_dice in num_dice_array:
            dist = results_exact[num_dice]
            if s >= len(dist) or dist[s] == 0:
                row_str += f"{'       -':>8} | "
            else:
                p = dist[s]
                if p < 0.005 and p > 0:
                     val_str = "<0.01%"
                     row_str += f"{val_str:>8} | "
                else:
                     val_str = f"{p:>.2f}%"
                     row_str += f"{val_str:>8} | "
        print(row_str)
        
    print("=" * len(header))
    print("\n")
    
    # --- CUMULATIVE PROBABILITIES TABLE (X OR MORE) ---
    print(f"Target Number: {target_number} (X OR MORE %)")
    print("=" * (11 + 11 * len(num_dice_array)))
    print(header)
    print("-" * len(header))
    
    # Print the cumulative rows
    for s in range(max_successes_possible + 1):
        row_str = f"{s:<8} | "
        for num_dice in num_dice_array:
            dist_cumul = results_cumulative[num_dice]
            dist_exact = results_exact[num_dice]
            
            if s >= len(dist_cumul) or dist_cumul[s] == 0:
                row_str += f"{'       -':>8} | "
            else:
                if s == 0:
                    # Formatting [00.00] for 0 successes representing exact percent
                    val_str = f"[{dist_exact[0]:05.2f}]"
                    row_str += f"{val_str:>8} | "
                else:
                    # Formatting normal percentages for cumulative X or more percent
                     p = dist_cumul[s]
                     if p < 0.005 and p > 0:
                         val_str = "<0.01%"
                         row_str += f"{val_str:>8} | "
                     else:
                         val_str = f"{p:>.2f}%"
                         row_str += f"{val_str:>8} | "
        print(row_str)
        
    print("=" * len(header))

In [15]:
# Example Usage 1: Fixed Target, Array of Dice Pools
NUM_DICE_ARRAY = [1,2,3,4,5,6,7,8,9,10]  # Compare columns for 2, 5, and 10 dice
TARGET_NUMBER = 19


calculate_success_distribution(NUM_DICE_ARRAY, TARGET_NUMBER)

Target Number: 19 (EXACT %)
Succes   |   1 Dice |   2 Dice |   3 Dice |   4 Dice |   5 Dice |   6 Dice |   7 Dice |   8 Dice |   9 Dice |  10 Dice | 
-------------------------------------------------------------------------------------------------------------------------
0        |    5.00% |    0.25% |    0.01% |   <0.01% |   <0.01% |   <0.01% |   <0.01% |   <0.01% |   <0.01% |   <0.01% | 
1        |   90.00% |    9.00% |    0.68% |    0.05% |   <0.01% |   <0.01% |   <0.01% |   <0.01% |   <0.01% |   <0.01% | 
2        |    5.00% |   81.50% |   12.19% |    1.22% |    0.10% |    0.01% |   <0.01% |   <0.01% |   <0.01% |   <0.01% | 
3        |        - |    9.00% |   74.25% |   14.71% |    1.83% |    0.18% |    0.02% |   <0.01% |   <0.01% |   <0.01% | 
4        |        - |    0.25% |   12.19% |   68.04% |   16.71% |    2.49% |    0.29% |    0.03% |   <0.01% |   <0.01% | 
5        |        - |        - |    0.68% |   14.71% |   62.71% |   18.26% |    3.16% |    0.42% |    0.05% |   <0.01%

***
## Fixed Dice Pool vs. Array of Target Numbers
The function below flips the variables: it takes a fixed number of dice, and calculates the success distribution against an array of target numbers.

In [16]:
import numpy as np

def calculate_targets_distribution(num_dice, target_array):
    if isinstance(target_array, int):
        target_array = [target_array]
        
    # Sort the array so columns go from lowest to highest target
    target_array = sorted([t for t in target_array if 2 <= t <= 20])
    
    if not target_array:
        print("No valid target numbers provided (must be 2-20).")
        return
    
    results_exact = {}
    results_cumulative = {}
    max_successes_possible = 0
    
    for target in target_array:
        # Probabilities for a single die based on this specific target
        p_0 = (20 - target) / 20.0     # Rolls > target
        p_1 = (target - 1) / 20.0      # Rolls < target
        p_2 = 1.0 / 20.0               # Roll == target
        
        single_die_poly = np.array([p_0, p_1, p_2])
        
        dist_poly = np.polynomial.polynomial.polypow(single_die_poly, num_dice)
        percentages = dist_poly * 100
        results_exact[target] = percentages
        
        # Compute cumulative probabilities (X successes OR BETTER)
        cumul = np.zeros_like(percentages)
        for s in range(len(percentages)):
            cumul[s] = np.sum(percentages[s:])
        results_cumulative[target] = cumul
        
        max_successes_possible = max(max_successes_possible, len(dist_poly) - 1)
        
    # --- EXACT PROBABILITIES TABLE ---
    print(f"Rolling {num_dice} Dice (EXACT %)")
    print("=" * (11 + 11 * len(target_array)))
    
    # Header
    header = f"{'Succes':<8} | "
    for target in target_array:
        h_col = f"TAR {target}"
        header += f"{h_col:>8} | "
        
    print(header)
    print("-" * len(header))
    
    # Print the exact rows
    for s in range(max_successes_possible + 1):
        row_str = f"{s:<8} | "
        for target in target_array:
            dist = results_exact[target]
            if s >= len(dist) or dist[s] == 0:
                row_str += f"{'       -':>8} | "
            else:
                p = dist[s]
                if p < 0.005 and p > 0:
                     val_str = "<0.01%"
                     row_str += f"{val_str:>8} | "
                else:
                     val_str = f"{p:>.2f}%"
                     row_str += f"{val_str:>8} | "
        print(row_str)
        
    print("=" * len(header))
    print("\n")
    
    # --- CUMULATIVE PROBABILITIES TABLE (X OR MORE) ---
    print(f"Rolling {num_dice} Dice (X OR MORE %)")
    print("=" * (11 + 11 * len(target_array)))
    print(header)
    print("-" * len(header))
    
    for s in range(max_successes_possible + 1):
        row_str = f"{s:<8} | "
        for target in target_array:
            dist_cumul = results_cumulative[target]
            dist_exact = results_exact[target]
            
            if s >= len(dist_cumul) or dist_cumul[s] == 0:
                row_str += f"{'       -':>8} | "
            else:
                if s == 0:
                    # Formatting [00.00] for 0 successes representing exact percent
                    val_str = f"[{dist_exact[0]:05.2f}]"
                    row_str += f"{val_str:>8} | "
                else:
                    # Formatting normal percentages for cumulative X or more percent
                     p = dist_cumul[s]
                     if p < 0.005 and p > 0:
                         val_str = "<0.01%"
                         row_str += f"{val_str:>8} | "
                     else:
                         val_str = f"{p:>.2f}%"
                         row_str += f"{val_str:>8} | "
        print(row_str)
        
    print("=" * len(header))

In [17]:
# Example Usage 2: Fixed Dice Pool, Array of Targets
FIXED_DICE = 5 
TARGET_ARRAY = [4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20]  # Compare target numbers 5, 10, and 15

calculate_targets_distribution(FIXED_DICE, TARGET_ARRAY)

Rolling 5 Dice (EXACT %)
Succes   |    TAR 4 |    TAR 5 |    TAR 6 |    TAR 7 |    TAR 8 |    TAR 9 |   TAR 10 |   TAR 11 |   TAR 12 |   TAR 13 |   TAR 14 |   TAR 15 |   TAR 16 |   TAR 17 |   TAR 18 |   TAR 19 |   TAR 20 | 
------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------
0        |   32.77% |   23.73% |   16.81% |   11.60% |    7.78% |    5.03% |    3.12% |    1.85% |    1.02% |    0.53% |    0.24% |    0.10% |    0.03% |    0.01% |   <0.01% |   <0.01% |        - | 
1        |   30.72% |   31.64% |   30.01% |   26.78% |   22.68% |   18.30% |   14.06% |   10.25% |    7.04% |    4.50% |    2.63% |    1.37% |    0.60% |    0.20% |    0.04% |   <0.01% |        - | 
2        |   21.76% |   24.79% |   27.44% |   29.18% |   29.70% |   28.91% |   26.88% |   23.81% |   20.00% |   15.81% |   11.61% |    7.75% |    4.54% |    2.17% |    0.73% |    

***
## 2D Matrix: Dice Pools vs Target Numbers
This function creates cross-comparison matrices. It outputs the exact percentage chance and cumulative percentage chance of rolling 0 to 10+ successes for every combination of dice pool and target number.

In [ ]:
import numpy as np

def calculate_matrix_comparison(num_dice_array, target_array):
    if isinstance(num_dice_array, int): num_dice_array = [num_dice_array]
    if isinstance(target_array, int): target_array = [target_array]
    
    num_dice_array = sorted(list(set(num_dice_array)))
    target_array = sorted([t for t in target_array if 2 <= t <= 20])
    
    if not target_array or not num_dice_array:
        print("Valid arrays for both targets and dice must be provided.")
        return
        
    col_width = 14
    
    # Header
    header = f"{'Dice':<8} | "
    for target in target_array:
        h_col = f"TAR {target}"
        header += f"{h_col:>{col_width}} | "
        
    sep = "-" * len(header)
    eq_sep = "=" * len(header)
    
    exact_distributions = {}
    cumul_distributions = {}
    
    for num_dice in num_dice_array:
        exact_distributions[num_dice] = {}
        cumul_distributions[num_dice] = {}
        for target in target_array:
            p_0 = (20 - target) / 20.0
            p_1 = (target - 1) / 20.0
            p_2 = 1.0 / 20.0
            
            single_die_poly = np.array([p_0, p_1, p_2])
            dist_poly = np.polynomial.polynomial.polypow(single_die_poly, num_dice)
            percentages = dist_poly * 100
            
            exact_vals = []
            cumul_vals = []
            
            for s in range(10):
                chance_exact = percentages[s] if s < len(percentages) else 0.0
                chance_cumul = np.sum(percentages[s:]) if s < len(percentages) else 0.0
                
                exact_vals.append((str(s), chance_exact))
                if s == 0:
                    cumul_vals.append((str(s), chance_exact))
                else:
                    cumul_vals.append((str(s), chance_cumul))
                    
            chance_10plus = np.sum(percentages[10:]) if 10 < len(percentages) else 0.0
            exact_vals.append(("10+", chance_10plus))
            cumul_vals.append(("10+", chance_10plus))
            
            exact_distributions[num_dice][target] = exact_vals
            cumul_distributions[num_dice][target] = cumul_vals
            
    def print_matrix(title, dist_dict, is_cumulative):
        print(title)
        print(eq_sep)
        print(header)
        print(sep)
        
        for num_dice in num_dice_array:
            for line_idx in range(11):
                if line_idx == 0:
                    row_str = f"{num_dice:<8} | "
                else:
                    row_str = f"{' ':<8} | "
                    
                for target in target_array:
                    label, chance = dist_dict[num_dice][target][line_idx]
                    
                    if is_cumulative and line_idx == 0:
                        val_str = f"[{chance:05.2f}]"
                    else:
                        if chance == 0:
                            val_str = "0.00%"
                        elif chance < 0.005 and chance > 0:
                            val_str = "<0.01%"
                        else:
                            val_str = f"{chance:.2f}%"
                            
                    visible_len = len(label) + 2 + len(val_str)
                    pad = " " * max(0, col_width - visible_len)
                    bold_label = f"\033[1m{label}\033[0m"
                    
                    row_str += f"{pad}{bold_label}: {val_str} | "
                print(row_str)
            print(sep)
        print(eq_sep)
        print("\n")
        
    print_matrix("Full Distribution Matrix (EXACT %, 0 to 10+ successes)", exact_distributions, False)
    print_matrix("Full Distribution Matrix (X OR MORE %, 0 to 10+ successes)", cumul_distributions, True)


In [ ]:
# Example Usage 3: 2D Matrix Combination
DICE_POOLS_TO_TEST = [1, 2, 3, 5, 10]
TARGETS_TO_TEST = [5, 10, 14, 15, 16, 20]

calculate_matrix_comparison(DICE_POOLS_TO_TEST, TARGETS_TO_TEST)


In [ ]:
# Optional: Interactive Sliders to compare two different dice pools
try:
    import ipywidgets as widgets
    from IPython.display import display, clear_output

    out = widgets.Output()

    def on_change(change):
        with out:
            clear_output(wait=True)
            # Sort lists and filter out 0 values if users drop dice pools to 0
            dice_vals = [d for d in [dice1_slider.value, dice2_slider.value, dice3_slider.value] if d > 0]
            if not dice_vals:
                dice_vals = [1] # fallback
            # Remove duplicates for cleaner columns
            dice_vals = list(set(dice_vals))
            calculate_success_distribution(dice_vals, target_slider.value)

    dice1_slider = widgets.IntSlider(value=3, min=0, max=50, description='Dice Pool 1:')
    dice2_slider = widgets.IntSlider(value=6, min=0, max=50, description='Dice Pool 2:')
    dice3_slider = widgets.IntSlider(value=9, min=0, max=50, description='Dice Pool 3:')
    target_slider = widgets.IntSlider(value=10, min=2, max=20, description='Target:')

    dice1_slider.observe(on_change, names='value')
    dice2_slider.observe(on_change, names='value')
    dice3_slider.observe(on_change, names='value')
    target_slider.observe(on_change, names='value')

    print("Interactive Comparison:")
    display(widgets.HBox([dice1_slider, dice2_slider, dice3_slider]))
    display(target_slider)
    display(out)

    # Initial render
    on_change(None)
except ImportError:
    pass